# 🔎 Reading Apache Spark Query Plans — From First Principles to Interview Ready

**A hands-on, self-contained PySpark learning notebook** based on your uploaded ~39-minute transcript and `2_reading_query_plans.ipynb`.

**Learning goals**
1. Explain **parsed → analyzed → optimized → physical** plans, Catalyst, and AQE.
2. Read plans **from scans (leaves) up to the final result (root)** and recognize common operators.
3. Predict when Spark **shuffles** for `repartition`, joins, `groupBy`, and `countDistinct`.
4. Diagnose predicate/projection pushdown and understand its limitations.
5. Explain trade-offs and defend your decisions in Spark/Databricks data-engineering interviews.

**How to study:** Run sections in order → predict each plan **before** calling `explain()` → inspect your output → speak the interview answer aloud → solve the exercises without looking at the solutions.

> 📌 **Version caveat:** Physical plan text is not a stable API. Operator names, stages, partitions, AQE changes and pushdown details can vary by Spark version, settings, data size and source. Use the *patterns* rather than memorize a specific screenshot.

## Start here — run this as a Python Jupyter notebook

This is a real **Python 3 `.ipynb` notebook**, not a PDF or notes-only guide. Open it in **JupyterLab**, **Jupyter Notebook**, **VS Code**, or **Google Colab**. Set the kernel to **Python 3**, then run the executable code cells from top to bottom.

**Prerequisites:** Apache Spark via `pyspark` and a compatible Java runtime. For a new local or Colab environment, run `%pip install "pyspark>=3.5,<5"` in a *separate* notebook cell if PySpark is not installed, and restart the kernel if needed. Spark and Java compatibility depends on the PySpark version.

**Learning flow:** Read a concept → predict its physical plan → run the Python cell → compare your prediction to `explain()` → answer the interview check → attempt the exercises before viewing the solutions.

**No external datasets needed:** The setup cells create `customers.parquet` and `transactions.parquet` in a temporary folder. These are illustrative replacements for the external files referenced by the original uploaded notebook.

In [ ]:
# Notebook health check: verifies that the Python kernel is active.
import sys
print("Python notebook kernel:", sys.executable)
print("Python version:", sys.version.split()[0])
# If the next setup cell reports that PySpark is missing, use the installation
# command given above, restart this kernel, and continue.

## Road map: transcript ↔ original notebook ↔ this lab

| Video timestamp | Concept from the transcript | Original notebook | This notebook |
|---|---|---|---|
| 01:35–04:45 | Query planning, Catalyst, pushdown | “Spark's Query Plans” | §2–3 |
| 04:52–09:00 | Narrow transformations | Narrow Transformations | §4 |
| 09:05–12:55 | Repartition, Exchange, round robin, AQE | Repartition | §5 |
| 13:03–17:33 | Coalesce | Coalesce | §6 |
| 17:33–23:25 | Sort-merge join, hash partitioning | Joins | §7 |
| 23:26–28:06 | `groupBy().count()` and `sum()` | GroupBy | §8 |
| 28:07–33:55 | `countDistinct()` per customer | GroupBy Count Distinct | §9 |
| 34:02–38:59 | Pushdown, residual filters, limitations | Interesting Observations | §10 |
| Extra interview prep | Join strategies, skew, AQE tuning, troubleshooting | Not in original | §11–15 |

**What changed:** The original notebook reads `../data/data_skew/*.parquet` and references an external diagram that were **not included** in the upload. Here we generate small data and write real Parquet files locally so **FileScan**/**PushedFilters** can be inspected without those missing assets. Source examples are preserved in spirit, including `cust_id`, `city`, `amt`, `age`, and `countDistinct(city)`.

## §1 — Setup and reproducible input data

### Requirements
- Python notebook with `pyspark` and Java configured (JDK version compatible with your Spark release).
- For a fresh local environment, install once in a separate notebook cell: `%pip install "pyspark>=3.5,<5"` and restart the kernel if needed.
- No Databricks-specific APIs, cloud credentials, missing external Parquet files, or internet connection are needed **after PySpark is installed**.

We'll deliberately use **few shuffle partitions** for readable plans; in production, choose them based on data size, parallelism and AQE.

In [ ]:
import sys
import tempfile
from pathlib import Path

try:
    import pyspark
    from pyspark.sql import SparkSession, functions as F, types as T
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'PySpark is required. Install with %pip install "pyspark>=3.5,<5" '
        'in a notebook cell, restart the kernel, and rerun.'
    ) from exc

spark = (SparkSession.builder
    .appName('ReadingSparkQueryPlans_InterviewLab')
    .master('local[2]')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.sql.adaptive.enabled', 'false')  # stable first-pass plans
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Python:', sys.version.split()[0], '| PySpark:', pyspark.__version__)
print('Shuffle partitions:', spark.conf.get('spark.sql.shuffle.partitions'))
print('AQE enabled:', spark.conf.get('spark.sql.adaptive.enabled'))

### Our tiny business scenario

A **customers** table (one record per customer) and **transactions** table (many transactions per customer). We include repeated cities, a missing customer match, and nulls to make joins and distinct counts meaningful.

**Important:** `age` is intentionally a **string**, matching the original notebook's cast/pushdown example. Data volume is deliberately small; this lab demonstrates plan *shapes*, not benchmark performance.

In [ ]:
customers_data = [
    ('C01', 'Ava Stone',  '30', 'F', '1994-06-10', 'boston'),
    ('C02', 'Ben Rivera', '48', 'M', '1976-02-19', 'boston'),
    ('C03', 'Cara Singh', '51', 'F', '1973-11-07', 'seattle'),
    ('C04', 'Dev Patel',  '37', 'M', '1987-09-23', 'chicago'),
    ('C05', 'Ella Park',  '62', 'F', '1962-12-02', 'boston'),
    ('C06', 'Finn Gray',  '28', 'M', '1996-08-12', 'seattle'),
    ('C07', 'Gia Young',  None, 'F', None,         'miami'),
    ('C08', 'Hugo Lee',   '45', 'M', '1979-04-05', 'chicago'),
]
customer_schema = T.StructType([
    T.StructField('cust_id', T.StringType()),
    T.StructField('name', T.StringType()),
    T.StructField('age', T.StringType()),
    T.StructField('gender', T.StringType()),
    T.StructField('birthday', T.StringType()),
    T.StructField('city', T.StringType()),
])
transactions_data = [
    ('T01','C01','boston',  40.0), ('T02','C01','boston',  60.0),
    ('T03','C01','chicago', 30.0), ('T04','C01','seattle', 20.0),
    ('T05','C02','boston',  95.0), ('T06','C02','boston',  12.0),
    ('T07','C02','miami',   25.0), ('T08','C03','seattle', 10.0),
    ('T09','C03','seattle', 15.0), ('T10','C03','boston',  22.0),
    ('T11','C04','chicago', 80.0), ('T12','C04','chicago', 17.0),
    ('T13','C04','miami',   50.0), ('T14','C05','boston',  30.0),
    ('T15','C05','miami',   45.0), ('T16','C05','seattle', 10.0),
    ('T17','C06','seattle', 33.0), ('T18','C06','seattle', 12.0),
    ('T19','C07','miami',   99.0), ('T20','C08','chicago', 13.0),
    ('T21','C08','boston',   8.0), ('T22','C08','boston',  12.0),
    ('T23','C08',None,     18.0), ('T24','C99','boston',  11.0),
    ('T25',None, 'miami',  21.0), ('T26','C03',None,      5.0),
]
transaction_schema = T.StructType([
    T.StructField('txn_id', T.StringType()),
    T.StructField('cust_id', T.StringType()),
    T.StructField('city', T.StringType()),
    T.StructField('amt', T.DoubleType()),
])
raw_customers = spark.createDataFrame(customers_data, customer_schema)
raw_transactions = spark.createDataFrame(transactions_data, transaction_schema)

lab_dir = Path(tempfile.mkdtemp(prefix='spark_query_plan_lab_'))
customers_file = str(lab_dir / 'customers.parquet')
transactions_file = str(lab_dir / 'transactions.parquet')
raw_customers.repartition(4).write.mode('overwrite').parquet(customers_file)
raw_transactions.repartition(4).write.mode('overwrite').parquet(transactions_file)

df_customers = spark.read.parquet(customers_file)
df_transactions = spark.read.parquet(transactions_file)
print('Lab files:', lab_dir)
print('Customers:', df_customers.count(), '| Transactions:', df_transactions.count())
df_customers.orderBy('cust_id').show(truncate=False)
df_transactions.orderBy('txn_id').show(8, truncate=False)

In [ ]:
print('CUSTOMER SCHEMA:')
df_customers.printSchema()
print('TRANSACTION SCHEMA:')
df_transactions.printSchema()

# Validate the tutorial's starting assumptions.
assert df_customers.count() == 8
assert df_transactions.count() == 26
assert dict(df_customers.dtypes)['age'] == 'string'

### Think before running

**Question:** Why write the synthetic data to Parquet and reread it instead of keeping only `spark.createDataFrame`?

**Answer:** Plans from an in-memory DataFrame show operators such as `ExistingRDD`. A Parquet scan exposes useful scan metadata such as `ReadSchema`, `DataFilters`, and potentially `PushedFilters`, which are central to this video.

## §2 — How Spark turns DataFrame code into an execution plan

**Video reference: 01:35–04:45.**

```text
DataFrame / SQL expression
           │
           ▼
  Parsed Logical Plan          ← syntax represented as a logical tree; names may be unresolved
           │
           ▼
  Analyzed Logical Plan        ← names/types checked against catalog & available schemas
           │
           ▼
  Optimized Logical Plan       ← Catalyst rewrites: prune columns, simplify predicates, etc.
           │
           ▼
  Physical Plan                ← specific operators: FileScan, HashAggregate, Exchange, SortMergeJoin
           │
           ▼
  Runtime execution            ← tasks on partitions; AQE may adapt supported query stages
```

### What to say in an interview

> Spark transformations build a lazy logical computation. The analyzer resolves attributes and data types. Catalyst rewrites the logical plan, and the physical planner selects physical operators. When an action triggers execution, Spark schedules stages and tasks; with AQE enabled, Spark may re-optimize parts of the plan using runtime statistics.

**Refinement to the transcript:** “syntax check → unresolved plan” is a helpful mental model, but the exact order and details depend on whether you use SQL or DataFrame APIs. `explain(True)` reports the **Parsed**, **Analyzed**, **Optimized**, and **Physical** plan views. Catalyst uses both rule-based optimization and supported statistics/cost-based choices; it does not blindly enumerate *every* possible physical plan.

In [ ]:
plan_demo = (df_customers
    .filter(F.col('city') == 'boston')
    .select('cust_id', 'name'))
plan_demo.explain(extended=True)

**Reading checklist:**
1. **Parsed:** Which DataFrame operations or SQL clauses appear? Are columns already resolved?
2. **Analyzed:** What types/attribute IDs do Spark attach? Does `city` exist?
3. **Optimized:** Are unused columns removed? Were filters moved closer to the scan?
4. **Physical:** Which scan, projection, filter, exchange, join and aggregate operators execute?

**What is laziness?** Calling `filter` or `select` builds a new DataFrame; calling `explain()` inspects its plan, while `show()`, `count()`, `collect()` or `write()` trigger jobs. Some metadata operations can cause work, so avoid treating *every* method as purely lazy.

In [ ]:
print('FORMATTED — easier operator-level reading:')
plan_demo.explain(mode='formatted')
print('\nCOST — statistics/estimates, when available:')
plan_demo.explain(mode='cost')

### Quick interview check

**Q:** Does `explain()` execute the complete DataFrame computation? **A:** No; it explains the plan. It is not a substitute for actual runtime metrics.

**Q:** Where are physical plans chosen? **A:** From an optimized logical plan during physical planning, with rules, strategies, available statistics and heuristics (plus adaptive changes at runtime when enabled).

## §3 — Operator dictionary: how to read physical plans

**Read the plan from the bottom leaves upward to the root.** Two join children mean there are two input branches, not a single linear pipeline.

| Operator / text | What it means | What to look for |
|---|---|---|
| `FileScan parquet` or `BatchScan` | Read source data | `ReadSchema`, `PushedFilters`, `PartitionFilters` |
| `ColumnarToRow` | Convert vectorized batches to row representation for downstream operators | Not always present; not necessarily bad |
| `Filter` | Apply a row predicate | May remain even if some predicate was pushed down |
| `Project` | Select or compute columns | Column pruning and expression computation |
| `Exchange` | Data movement / physical distribution boundary | `RoundRobinPartitioning`, `hashpartitioning`, `SinglePartition` |
| `Sort` | Sort within partitions | Common before `SortMergeJoin` |
| `SortMergeJoin` | Shuffle/sort-compatible equi-join | Two branches, often two exchanges and sorts |
| `BroadcastExchange` | Replicate a small relation to tasks | Often feeds `BroadcastHashJoin` |
| `BroadcastHashJoin` | Probe hash table built from broadcast side | Often avoids shuffling the large side |
| `HashAggregate` | Aggregate group values | `partial_*` below exchange; final above |
| `Coalesce` | Reduce the number of partitions without introducing a shuffle itself | May lower task parallelism |
| `AdaptiveSparkPlan` | AQE wrapper around a plan | `isFinalPlan=false/true` can depend on observation point |
| `WholeStageCodegen` / `*(n)` | Fused code-generated execution pipeline | Asterisk is **not** a shuffle marker |

**One-line mental model:** Scan → prune/filter → (maybe Exchange) → sort/join/aggregate → project result.

## §4 — Narrow transformations: `filter`, `withColumn`, `select`

**Video reference: 04:52–09:00.** A narrow dependency means each output partition depends on a limited set of input partitions and **does not require a full data redistribution**. Filter, select and simple column expressions are common examples.

**Original notebook sequence:** `city == 'boston'` → split full name → add 5 to age → select a few columns.

**Predict:** Should there be an `Exchange` introduced by these transformations? Which columns must the Parquet reader load?

In [ ]:
df_narrow_transform = (df_customers
    .filter(F.col('city') == 'boston')
    .withColumn('first_name', F.split(F.col('name'), ' ').getItem(0))
    .withColumn('last_name', F.split(F.col('name'), ' ').getItem(1))
    .withColumn('age', F.col('age').cast('int') + F.lit(5))
    .select('cust_id', 'first_name', 'last_name', 'age', 'gender', 'birthday'))

df_narrow_transform.orderBy('cust_id').show(truncate=False)
df_narrow_transform.explain(mode='extended')

### Decode the plan

- `FileScan parquet`: read the customer file; `ReadSchema` should omit source columns no longer needed (for example, not every possible field is guaranteed to be read).
- `Filter`: keep `boston` rows, commonly combined with a generated `isnotnull(city)` check.
- `Project`: Spark can combine several column-expression operations and the final selection into a compact projection.
- **No new shuffle** due to `filter`, `withColumn`, or `select` themselves.

**Two different optimizations:**
- **Predicate pushdown:** send supported row filters toward a data source, sometimes reducing file/row-group work.
- **Projection pushdown / column pruning:** avoid reading unused columns. This is especially valuable with columnar Parquet.

**Interview trap:** A cast of a source string field to an integer is not automatically guaranteed to be pushed into Parquet; Spark may evaluate it after scanning. Type-aware schemas are a practical performance win.

In [ ]:
# Explicitly inspect just the scan-related information in a formatted explanation.
df_narrow_transform.explain(mode='formatted')

**Self-test:** Why might `Project` appear as **one** operator even though we wrote multiple `withColumn` calls?

<details><summary>Answer</summary> Catalyst can collapse compatible projections and simplify expressions. DataFrame code is a logical recipe, not a guaranteed one-to-one list of runtime operators. </details>

## §5 — Repartition: a wide transformation and a shuffle

**Video reference: 09:05–12:55.** In the original, `repartition(24)` produces an `Exchange` with `RoundRobinPartitioning(24)` (depending on Spark release and surrounding plan).

A **wide** transformation may require records from many upstream partitions to be delivered to downstream partitions. This movement is a **shuffle**, and often means network IO, serialization, disks and a stage boundary.

```text
before: P0  P1  P2  P3
          \  |  |  /      Exchange: round-robin-style data redistribution
           \ | / /       ▼
after:  Q0  Q1  Q2  Q3  Q4  Q5
```

**Important:** Round robin here is the **distribution strategy**, not a promise that individual rows will be assigned in a particular deterministic order.

In [ ]:
print('Current Parquet read partitions:', df_transactions.rdd.getNumPartitions())
rr = df_transactions.repartition(6)
print('repartition(6) partitions:', rr.rdd.getNumPartitions())
rr.explain(mode='extended')

### Round robin versus hash partitioning

- `df.repartition(6)` usually redistributes rows across 6 partitions using a round-robin-style partitioner; `Exchange RoundRobinPartitioning(6)` is the signature.
- `df.repartition(6, 'cust_id')` uses **hash partitioning by key**, so equal keys are routed to the same target partition (barring later changes to the plan).
- Repartitioning by a key is useful before **some** keyed workloads, but Spark may still choose another exchange if the exact required distribution does not match downstream needs.

In [ ]:
by_customer = df_transactions.repartition(4, 'cust_id')
by_customer.explain(mode='formatted')

# Observational demonstration: each non-null cust_id maps to one partition.
key_partition = (by_customer
    .select('cust_id', F.spark_partition_id().alias('partition_id'))
    .groupBy('cust_id')
    .agg(F.countDistinct('partition_id').alias('num_partitions_for_key')))
key_partition.orderBy('cust_id').show(truncate=False)
assert key_partition.filter(F.col('num_partitions_for_key') != 1).count() == 0

### Hash partitioning: intuition (not Spark's literal hash implementation)

For `N` shuffle partitions, picture `target_partition = positive_hash(key) % N`. If every `C01` hashes to `2`, all `C01` records land in the same target partition (for that specific exchange).

**Nuances:** Spark uses its own hash functions and key encodings; Python's `hash()` is not the Spark partitioner. Equal keys co-locate but **different** keys can share a partition. Hash partitioning does not sort rows.

**Q:** Does `repartition(6)` guarantee a performance improvement? **A:** No. It costs a shuffle, and 6 may be too many/too few. Repartition for a known distribution or parallelism need, then measure.

## §6 — Coalesce: fewer partitions, usually no shuffle

**Video reference: 13:03–17:33.** `df.coalesce(n)` creates fewer partitions by *grouping existing parent partitions*. It does **not** introduce a new shuffle. The physical operator is usually `Coalesce n` rather than an `Exchange`.

```text
before: P0  P1  P2  P3  P4  P5
          \ /    \ /    \ /
after:     Q0     Q1     Q2         (narrow dependency)
```

### ⚠️ Source clarification worth memorizing for interviews

The transcript speculates that aggressive `coalesce(2)` may itself require a network shuffle, depending on executor placement. **Do not use that as the general Spark DataFrame rule.** Official Spark API documentation describes DataFrame `coalesce()` as a **narrow dependency without a shuffle**, including aggressive reductions. Tasks may still read parent partition data across hosts through normal scheduling, but that is **not equivalent to `Exchange` shuffle**. An upstream shuffle can, of course, still be present in the full plan.

Also, `coalesce()` does **not** necessarily preserve a useful hash partitioning guarantee; do not claim the partitioning scheme is always unchanged.

In [ ]:
coalesced = df_transactions.coalesce(2)
print('Starting partitions:', df_transactions.rdd.getNumPartitions())
print('After coalesce(2):', coalesced.rdd.getNumPartitions())
coalesced.explain(mode='extended')

print('\nContrast with repartition(2):')
df_transactions.repartition(2).explain(mode='formatted')

| Interview comparison | `repartition(n)` | `coalesce(n)` |
|---|---|---|
| Adds a shuffle itself? | **Yes**, normally | **No** |
| Increase partitions? | Yes | Cannot increase beyond current count |
| Decrease partitions? | Yes | Yes |
| Balance data across new partitions? | Typically better | Not guaranteed; may preserve imbalance |
| Main risk | Expensive shuffle | Too few tasks / poor parallelism |
| Typical use | Redistribute/skew workaround/key grouping | Reduce output file/task count modestly |

**Interview example:** Why can `coalesce(1)` be slow? All output work after coalescing can become one task, limiting parallelism. Do **not** blindly use it to create a single output file for large datasets.

In [ ]:
# A subtle but important difference: an earlier shuffle does not vanish.
upstream_shuffle_then_coalesce = df_transactions.repartition(6).coalesce(2)
upstream_shuffle_then_coalesce.explain(mode='formatted')
# Look for both an Exchange (from repartition) and Coalesce (from coalesce).

## §7 — Joins: SortMergeJoin, Exchange, Sort and broadcast

**Video reference: 17:33–23:25.** The transcript turns off automatic broadcast so the join can be studied as a **sort-merge join**.

### Sort-merge join mental model

```text
transactions: FileScan → Filter(not-null key) → Exchange(hash key) → Sort(key) ─┐
                                                                               ├→ SortMergeJoin → Project
customers:    FileScan → Filter(not-null key) → Exchange(hash key) → Sort(key) ─┘
```

1. Shuffle both sides on compatible `cust_id` partitioning (unless distribution is already satisfied).
2. Sort both sides by the join key within their partitions.
3. Merge matching sorted keys, producing the joined rows.

**Remember:** This is a common pattern, not a mandatory operator sequence for *every* join or input.

In [ ]:
# Disable both automatic and adaptive broadcast just for this demonstration.
previous_broadcast_threshold = spark.conf.get('spark.sql.autoBroadcastJoinThreshold')
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')

smj = (df_transactions.alias('t')
    .join(df_customers.alias('c'), on='cust_id', how='inner')
    .select('cust_id', F.col('t.txn_id'), F.col('t.city').alias('transaction_city'),
            F.col('c.name'), F.col('t.amt')))
smj.orderBy('txn_id').show(10, truncate=False)
smj.explain(mode='formatted')

### How to narrate this in an interview

> “I start at both input scans, check key null filtering and projected columns, then look for `Exchange hashpartitioning(cust_id, N)` on each side, a `Sort` on each key, and the final `SortMergeJoin`. Those exchanges indicate shuffle costs. I would check Spark UI metrics for shuffle read/write, spill, task skew and join cardinality before tuning.”

**Why `isnotnull(cust_id)` may appear even if you did not write it:** For an **inner equality join**, rows with null join keys cannot match, so Spark can often insert null filters. Different join types and null-safe equality have different semantics.

**Duplicate `city` trap:** After a join, both relations have `city`. Alias and explicitly project e.g. `transaction_city` versus `customer_city` rather than relying on an ambiguous `.select('city')`.

In [ ]:
# A forced broadcast hint works even when automatic broadcast is disabled.
bhj = (df_transactions.alias('t')
    .join(F.broadcast(df_customers).alias('c'), on='cust_id', how='inner')
    .select('cust_id', 'txn_id', F.col('c.name').alias('customer_name')))
bhj.explain(mode='formatted')
print('Same join row count?', smj.count() == bhj.count())
assert smj.count() == 24  # 26 transactions minus orphan C99 and null cust_id

### Compare join types and semantics

| Strategy | Typical signature | Key trade-off |
|---|---|---|
| Sort-merge | `SortMergeJoin`, often hash `Exchange` and `Sort` on both inputs | Handles large equi-join sides; shuffle/sort may be costly |
| Broadcast-hash | `BroadcastExchange`, `BroadcastHashJoin` | Avoid shuffling large probe side when other side is small enough; memory/broadcast cost |
| Shuffled-hash | `ShuffledHashJoin` | Shuffles keys; hash table built per partition; depends on statistics/config |
| Nested-loop | `BroadcastNestedLoopJoin` or Cartesian variants | Sometimes necessary for non-equi conditions; potentially very expensive |

**Q:** Should you disable broadcast joins in production? **A:** No. We disable them **only for explaining** sort-merge plans. Normally let the optimizer/AQE decide or use hints carefully when evidence justifies them.

In [ ]:
# Verify inner-join null/orphan behavior and make an outer-join comparison.
print('Rows before join:', df_transactions.count())
print('Rows after inner join:', smj.count())
print('Rows after left join:', df_transactions.join(df_customers, 'cust_id', 'left').count())

# Restore the setting before moving on.
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', previous_broadcast_threshold)

## §8 — GroupBy COUNT and SUM: local + global aggregation

**Video references: 23:26–28:06.** The original groups transactions by `city`, first to count transactions and then to sum `amt`.

```text
Input partitions  →  partial HashAggregate(city)  →  Exchange(hash city)
                 →  final HashAggregate(city)    →  city-level result
```

**Why partial aggregation matters:** Spark first combines records with the same key **within each input partition**. Shuffling partial totals (often much fewer rows) is cheaper than shuffling every raw input row.

Example local counts:
- Partition A: `boston=3, seattle=1`
- Partition B: `boston=2, seattle=4`
- Final: `boston=5, seattle=5`.

In [ ]:
df_city_counts = df_transactions.groupBy('city').count()
df_city_counts.orderBy('city').show(truncate=False)
df_city_counts.explain(mode='formatted')

# Self-check: every row, including NULL city, contributes to count().
assert sum(row['count'] for row in df_city_counts.collect()) == 26

In [ ]:
df_txn_amt_city = (df_transactions
    .groupBy('city')
    .agg(F.sum('amt').alias('txn_amt')))
df_txn_amt_city.orderBy('city').show(truncate=False)
df_txn_amt_city.explain(mode='formatted')

**Look for** a `partial_count` or `partial_sum` aggregation *before* an `Exchange`, and a final aggregate *after* it. Exact function labels vary.

**Interview Q:** Does every aggregation have to shuffle? **A:** A global keyed aggregation generally needs compatible key distribution and often shuffles; if the input already satisfies the required distribution or the optimizer uses a different arrangement, a new exchange may be avoidable. A simple local partial aggregate itself is narrow.

**Subtlety:** `count()` after `groupBy(city)` counts rows even when `city` is NULL (the NULL group exists). `countDistinct(city)` ignores null `city` values within each group; it answers a different question.

## §9 — `countDistinct(city)` by customer: why the plan is more complex

**Video reference: 28:07–33:55.** The original **code** is:

```python
(df_transactions.groupBy('cust_id')
    .agg(F.countDistinct('city').alias('city_count')))
```

The narration briefly says “group by city / distinct of city”, but the later explanation and original notebook **correctly group by `cust_id`**. The question is: **For each customer, in how many different non-null cities did they transact?**

### Logical reasoning (simplified, not a promise of exact physical steps)

1. Deduplicate `(cust_id, city)` combinations inside input partitions.
2. Bring matching `(cust_id, city)` pairs together; deduplicate globally.
3. Count one entry per distinct non-null city for each `cust_id`.
4. Ensure totals for the same customer are combined in a final aggregation.

Spark frequently uses **more aggregate/exchange steps** for exact distinct aggregates than for `count()` and `sum()`; how many exchanges you actually see depends on engine release, distinct expressions, partitioning and optimizer rules.

In [ ]:
df_distinct_city_counts = (df_transactions
    .groupBy('cust_id')
    .agg(F.countDistinct('city').alias('city_count')))
df_distinct_city_counts.orderBy('cust_id').show(truncate=False)
df_distinct_city_counts.explain(mode='formatted')

In [ ]:
# A concrete explanation with C01: 4 transactions, but only 3 distinct cities.
(df_transactions
    .filter(F.col('cust_id') == 'C01')
    .orderBy('txn_id')
    .show(truncate=False))

c01 = (df_distinct_city_counts
    .filter(F.col('cust_id') == 'C01')
    .first()['city_count'])
print('Distinct non-null cities for C01:', c01)
assert c01 == 3

# Nulls in city are not counted by countDistinct(city).
assert df_distinct_city_counts.filter("cust_id = 'C08'").first()['city_count'] == 2

### `countDistinct` versus `approx_count_distinct`

- `countDistinct(city)` gives an **exact** answer and can require expensive deduplication/shuffles.
- `approx_count_distinct(city)` uses an approximate algorithm and can be preferable for large, high-cardinality data when a known, controlled error is acceptable.
- Do not switch to approximate results silently in finance/audit or when exact counts are required.

**Interview answer (30 seconds):** “Exact count-distinct is harder than sum/count because duplicates may span partitions. Spark needs to coordinate the distinct keys across partitions before producing group counts. I inspect its aggregate and exchange stages, null handling, cardinality and memory pressure; approximate distinct is an option only if the business accepts an approximate answer.”

In [ ]:
# OPTIONAL extension: compare the two types of distinct count.
(df_transactions.groupBy('cust_id')
    .agg(F.countDistinct('city').alias('exact'),
         F.approx_count_distinct('city').alias('approx'))
    .orderBy('cust_id')
    .show(truncate=False))

## §10 — Predicate pushdown, column pruning, and why a Filter may remain

**Video references: 34:02–38:59.** Use the scan-level details to see what Spark **attempts to push down**, then distinguish it from the filter that the Spark engine itself evaluates.

| Scan field | Meaning |
|---|---|
| `PushedFilters` | Predicates represented for potential execution by the source reader |
| `DataFilters` | Predicates related to row/data filtering for the scan |
| `PartitionFilters` | Predicates used to skip partition directories (if input is partitioned by columns) |
| `ReadSchema` | Fields requested from the file reader (column pruning) |

**Caution:** Seeing `PushedFilters` does **not** prove that every row was filtered inside the Parquet reader. It can allow Parquet page/row-group pruning or other optimizations, and Spark may retain a **residual `Filter`** for correctness. Pushdown *support* is data-source- and expression-dependent.

In [ ]:
# Baseline: simple scalar predicate is a good pushdown candidate.
scalar_filter = (df_customers
    .filter(F.col('city') == 'boston')
    .select('cust_id', 'city'))
scalar_filter.explain(mode='formatted')
scalar_filter.orderBy('cust_id').show(truncate=False)

### Why do we still see `Filter` after pushdown?

**Interview answer:** Because the datasource pushdown does not necessarily implement the entire Spark SQL predicate with identical guarantees. The physical plan can keep a residual filter for semantic correctness even when scan-level pushdown helps reduce input. Think of the two mechanisms as **cooperating**, not mutually exclusive.

**Source refinement:** It is too strong to claim that every pushed condition is **always** evaluated twice, or that logical-versus-physical planning alone completely explains it. The source API's capabilities and expression semantics matter.

### Example 1: filter on a cast of the stored string `age`

The original notebook casts `age` to integer before testing if it is greater than 50. Because the actual Parquet field is a **string**, the reader typically cannot directly push the *numeric cast comparison* as an ordinary raw Parquet field predicate.

**Predict:** You may see null-related pushdown on the original field, while the cast comparison stays in Spark's `Filter`. This is illustrative, not guaranteed for every source/version.

In [ ]:
cast_filter = (df_customers
    .filter(F.col('age').cast('int') > F.lit(50))
    .select('cust_id', 'age'))
cast_filter.explain(mode='formatted')
cast_filter.orderBy('cust_id').show(truncate=False)

### Example 2: map element filter

Parquet's simple scalar filters differ from expressions involving `map_col['key']`. A map-element predicate is generally **not** pushed down as a basic Parquet scalar comparison; examine the scan's `PushedFilters` rather than assuming.

In [ ]:
map_schema = T.StructType([
    T.StructField('id', T.StringType()),
    T.StructField('properties', T.MapType(T.StringType(), T.StringType())),
])
map_rows = [
    ('A', {'eye': 'brown', 'hair': 'black'}),
    ('B', {'eye': 'blue', 'hair': 'brown'}),
    ('C', {'eye': 'brown', 'hair': 'red'}),
]
map_path = str(lab_dir / 'map_example.parquet')
spark.createDataFrame(map_rows, map_schema).write.mode('overwrite').parquet(map_path)
map_df = spark.read.parquet(map_path)
map_filtered = map_df.filter(F.col('properties').getItem('eye') == 'brown')
map_filtered.explain(mode='formatted')
map_filtered.orderBy('id').show(truncate=False)

### Bonus: Parquet partition pruning is *different* from predicate pushdown

When a dataset is physically written as `.../city=boston/...`, Spark can often skip other partition **directories** based on `PartitionFilters`. This is **partition pruning**, not the same as a `PushedFilters` entry applied within files.

In [ ]:
partitioned_path = str(lab_dir / 'transactions_by_city')
df_transactions.write.mode('overwrite').partitionBy('city').parquet(partitioned_path)
partitioned = spark.read.parquet(partitioned_path)
partitioned.filter(F.col('city') == 'boston').select('cust_id', 'amt').explain(mode='formatted')

**Troubleshooting pushdown:** If a slow query reads too much data, inspect `ReadSchema` first, then partition pruning, `PushedFilters`, file sizes and selectivity. Writing a filter in code does **not** guarantee an equivalent source-level filter.

## §11 — Adaptive Query Execution (AQE): planning can change at runtime

**Video reference: 11:46–12:55.** In the video, `AdaptiveSparkPlan isFinalPlan=false` means Spark has an **initial adaptive plan** that may be modified after it learns actual stage statistics.

AQE can (when applicable and configured):
- Coalesce small **post-shuffle** partitions.
- Replace sort-merge with a broadcast join if a side turns out small enough.
- Mitigate some skewed joins via partition splitting and related techniques.

**Two different `coalesce`s:** `df.coalesce(2)` is the *user-written DataFrame operation*. AQE's **post-shuffle partition coalescing** is a *runtime planner optimization*; they are not the same API call.

**Interview warning:** `isFinalPlan=false` means the *displayed plan view is not the final adaptive plan*, not that the DataFrame code is malformed or that Spark will definitely change the plan.

In [ ]:
print('Before:', spark.conf.get('spark.sql.adaptive.enabled'))
spark.conf.set('spark.sql.adaptive.enabled', 'true')

# Construct this query after enabling AQE; inspect initial and after an action.
aqe_demo = df_transactions.groupBy('cust_id').agg(F.sum('amt').alias('total'))
print('AQE-aware plan (before action):')
aqe_demo.explain(mode='formatted')
print('Action result:', aqe_demo.count())
print('Plan view after action (behavior differs by Spark version/call site):')
aqe_demo.explain(mode='formatted')

spark.conf.set('spark.sql.adaptive.enabled', 'false')  # restore predictable plans

**What to inspect in a real job:** Spark UI → SQL query → physical plan and stage metrics. Compare initial and final adaptive plans, runtime row counts, shuffle bytes, skewed task duration and spill. Explain text alone cannot tell you how many bytes actually moved.

## §12 — Troubleshooting case study: a Spark job is slow

**Interview prompt:** “Your daily transaction report slows from 4 minutes to 40 minutes after the table grows. How do you use the query plan?”

A strong sequence:
1. Confirm **what changed**: input size, code, table layout, Spark version, configurations, skew, late data.
2. Read the physical plan **bottom-up**. Identify repeated `Exchange`, `Sort`, huge scans and nonselective filters.
3. Inspect Spark UI **actual metrics**, not only the static plan: shuffle read/write, spilled bytes, GC, stage time, uneven task sizes.
4. Ask whether **projection pruning, partition pruning, or predicate pushdown** are effective.
5. Check **join strategy/cardinality**, skewed keys, AQE enabled, broadcast memory feasibility.
6. Change **one variable at a time**, rerun with the same representative workload, compare wall time and data correctness.

### A deliberately skewed example

If a single `cust_id` represents most transactions, hash partitioning sends its rows to the *same* target partition for a given exchange. More partitions alone may not split that hot key; skew-aware strategies may be needed.

In [ ]:
# Small, illustrative skew dataset (not a performance benchmark).
skew_rows = [('HOT', i) for i in range(50)] + [(f'C{i:02d}', 100 + i) for i in range(10)]
skew_df = spark.createDataFrame(skew_rows, ['cust_id', 'event_id'])
print('Row counts by key:')
skew_df.groupBy('cust_id').count().orderBy(F.desc('count')).show(5, False)

print('Planned distribution when partitioning by cust_id:')
skew_df.repartition(4, 'cust_id').explain(mode='formatted')

**Solution options with trade-offs:** Broadcast a genuinely small join side; ensure statistics exist; let AQE split eligible skewed join partitions; consider **salting** hot keys only with a carefully correct join/aggregation design; pre-aggregate when valid; reduce unnecessary scans/columns; avoid premature cache or arbitrary repartitioning. Always check result correctness and memory.

## §13 — Hands-on exercises (solve before looking at the key)

These tasks use the exact DataFrames from the lab. For each, **predict** the interesting physical operators, run `explain(mode='formatted')`, and then validate the result.

### Exercise A — Filter + projection (easy)

Filter transactions for `city == 'seattle'`, select only `cust_id`, `amt`, and examine whether the plan uses `ReadSchema` pruning and a scan-level pushed filter.

In [ ]:
# YOUR TURN — replace the right-hand side and rerun.
exercise_a = None  # df_transactions.filter(...).select(...)
# When ready: exercise_a.explain(mode='formatted'); exercise_a.show()

### Exercise B — Exchange versus Coalesce (easy)

Create 3 DataFrames: (1) original transactions repartitioned into 5 partitions, (2) original coalesced to 1, (3) repartitioned to 5 then coalesced to 2. **Explain which plan contains Exchange and why.**

In [ ]:
exercise_b1 = None
exercise_b2 = None
exercise_b3 = None
# Predict which show Exchange, then call .explain(mode='formatted') on each.

### Exercise C — Join choices (medium)

Join transactions to customers by `cust_id`, return `txn_id`, `cust_id`, `name`. Compare an unhinted plan with a **broadcast** hint on customers. Say when broadcast would be unsafe.

In [ ]:
exercise_c_default = None
exercise_c_broadcast = None

### Exercise D — Aggregation (medium)

Return `city`, `txn_count`, `total_amt` from transactions. Why may a partial `HashAggregate` appear before a shuffle?

In [ ]:
exercise_d = None

### Exercise E — Exact distinct count (medium)

For each customer, count **distinct non-null** transaction cities; check `C01 == 3`, `C08 == 2`. Explain why the plan differs from a sum.

In [ ]:
exercise_e = None

### Exercise F — Diagnose a pushdown failure (advanced)

Compare `df_customers.filter(F.col('age') > '50')` to a **numeric** comparison after `.cast('int')`. Are results and source-level pushed filters necessarily equivalent? **Hint:** lexicographic strings are not numeric comparisons.

In [ ]:
exercise_f_string = None
exercise_f_numeric = None

## §14 — Exercise solutions and assertions

Try the previous tasks first. These cells generate clean reference solutions without relying on your `exercise_*` variables.

In [ ]:
answer_a = (df_transactions
    .filter(F.col('city') == 'seattle')
    .select('cust_id', 'amt'))
answer_a.explain(mode='formatted')
assert answer_a.count() > 0

In [ ]:
answer_b1 = df_transactions.repartition(5)             # introduces Exchange
answer_b2 = df_transactions.coalesce(1)               # no new Exchange
answer_b3 = df_transactions.repartition(5).coalesce(2)  # Exchange upstream + Coalesce
for name, dataframe in [('repartition',answer_b1),
                        ('coalesce',answer_b2),
                        ('repartition+coalesce',answer_b3)]:
    print('\n', name)
    dataframe.explain(mode='formatted')

In [ ]:
answer_c_default = (df_transactions.alias('t')
    .join(df_customers.alias('c'), 'cust_id', 'inner')
    .select('txn_id','cust_id', F.col('c.name').alias('name')))
answer_c_broadcast = (df_transactions.alias('t')
    .join(F.broadcast(df_customers).alias('c'), 'cust_id', 'inner')
    .select('txn_id','cust_id', F.col('c.name').alias('name')))
print('DEFAULT PLAN:')
answer_c_default.explain(mode='formatted')
print('BROADCAST PLAN:')
answer_c_broadcast.explain(mode='formatted')
assert answer_c_default.count() == answer_c_broadcast.count() == 24

In [ ]:
answer_d = (df_transactions.groupBy('city')
    .agg(F.count('*').alias('txn_count'),
         F.sum('amt').alias('total_amt')))
answer_d.explain(mode='formatted')
answer_d.orderBy('city').show(truncate=False)
assert answer_d.agg(F.sum('txn_count')).first()[0] == 26

In [ ]:
answer_e = (df_transactions.groupBy('cust_id')
    .agg(F.countDistinct('city').alias('city_count')))
answer_e.explain(mode='formatted')
actual_e = {r['cust_id']: r['city_count'] for r in answer_e.collect()}
assert actual_e['C01'] == 3 and actual_e['C08'] == 2
print('Verified:', actual_e)

In [ ]:
answer_f_string = df_customers.filter(F.col('age') > '50')
answer_f_numeric = df_customers.filter(F.col('age').cast('int') > 50)
print('STRING comparison:')
answer_f_string.explain(mode='formatted')
print('NUMERIC comparison:')
answer_f_numeric.explain(mode='formatted')
print('String comparison IDs:', sorted(r.cust_id for r in answer_f_string.select('cust_id').collect()))
print('Numeric comparison IDs:', sorted(r.cust_id for r in answer_f_numeric.select('cust_id').collect()))
# With current sample values, results happen to coincide for >50; do not assume equivalence.
# Compare '9' with '50': '9' > '50' lexicographically, 9 > 50 is false.
print("Concept check: '9' > '50' =", '9' > '50', '| 9 > 50 =', 9 > 50)

**Solution reasoning to say aloud:**

- **A:** Filter and projection normally remain narrow; scalar Parquet pushdown may appear, and the reader should request fewer fields.
- **B:** Repartition shuffles, coalesce narrows, chained operations show both operators; an upstream shuffle doesn't make coalesce itself a shuffle.
- **C:** `BroadcastHashJoin` avoids a large-side shuffle but can exhaust memory or add costly broadcasts if the “small” side is too large.
- **D:** Local partial aggregates shrink the data sent over the network.
- **E:** Exact distinct must resolve cross-partition duplicates and ignores null city values.
- **F:** Changing the stored type/expression can alter semantics and whether a data-source pushdown is supported. Never replace a numeric predicate with a string comparison merely to obtain pushdown.

## §15 — Interview question bank (with concise model answers)

Use these as **flashcards**. Hide the answer with your hand, speak for 20–40 seconds, and give one trade-off.

### Foundations (Q1–Q8)

**1. What are parsed, analyzed, optimized, and physical plans?**  
Parsed represents the expression tree; analyzed resolves attributes/types; optimized applies Catalyst rewrites; physical chooses executable operators.

**2. Catalyst vs AQE?**  
Catalyst plans/optimizes before execution; AQE adapts eligible physical stages based on runtime statistics.

**3. Are DataFrame transformations lazy?**  
Most transformations construct the logical computation; actions such as `count` or `write` trigger execution.

**4. How do you inspect a plan?**  
`df.explain(True)` for all plan stages; `df.explain(mode='formatted')` for readable physical operators; `mode='cost'` for estimates.

**5. Why read plans bottom-up?**  
Inputs/scans are the leaves, and each parent operator consumes child output. For joins, analyze both branches.

**6. `Filter` vs `PushedFilters`?**  
`Filter` is Spark's row predicate; `PushedFilters` is eligible source-reader pushdown. Both may be present.

**7. What is column pruning?**  
Eliminating unused read columns, visible in `ReadSchema` on file scans.

**8. What does `*(1)` mean?**  
Whole-stage code generation pipeline marker, **not** a shuffle or one partition.

### Shuffles and joins (Q9–Q16)

**9. Define a narrow dependency.**  
An output partition depends on a bounded set of parent partitions without a full repartitioning shuffle; common examples are filter and select.

**10. What signals a shuffle?**  
Most often an `Exchange` operator. Check partitioning scheme, stage metrics, and whether the exchange is reused.

**11. `repartition` versus `coalesce`?**  
Repartition usually shuffles and can raise/lower partition counts; coalesce reduces counts without introducing a new shuffle, risking lower parallelism.

**12. What does `hashpartitioning(cust_id, N)` ensure?**  
Identical keys are assigned to the same target partition for that exchange; it does not guarantee equal partition sizes or sorted rows.

**13. Why does sort-merge join shuffle and sort?**  
It usually distributes matching keys together and sorts both sides by join key so matching runs can be merged efficiently.

**14. When favor broadcast hash join?**  
One side is small enough to replicate safely, allowing large-side processing without a full key-based shuffle.

**15. Why is `isnotnull(join_key)` injected?**  
For ordinary inner equality joins, null keys cannot match, so filtering null keys is often safe and useful. Not universal for all joins.

**16. What causes skew, and why can repartitioning fail to solve it?**  
A hot key can dominate one hash partition; increasing partition count does not split a single key by itself. Measure task imbalance.

### Aggregates, pushdown, tuning (Q17–Q26)

**17. Why is `HashAggregate` sometimes repeated?**  
Spark can partially aggregate per input partition and finish after distributing matching keys together.

**18. Why is `countDistinct` harder than `count`?**  
Duplicates can exist across partitions, so global deduplication is required before an exact per-group distinct count.

**19. Does `countDistinct(city)` include NULL?**  
No. Spark SQL's count over a column excludes nulls; `groupBy(city).count()` still forms a null group.

**20. Why might a pushed-down predicate still have a `Filter`?**  
The engine may retain a residual filter to guarantee full SQL semantics if source pushdown is partial or approximate.

**21. Why might `cast(age as int) > 50` not push into Parquet?**  
The stored field may be a string and the reader may not support expression evaluation with that cast.

**22. How does directory partition pruning differ from pushdown?**  
It skips files/directories using partition columns; row/data pushdown acts inside a source reader.

**23. What does `AdaptiveSparkPlan isFinalPlan=false` tell you?**  
That the current plan view is not the final adaptive plan. It may evolve when runtime stats become available.

**24. Why can `coalesce(1)` hurt performance?**  
It may collapse work to a single downstream task, reducing parallel execution and creating an output bottleneck.

**25. What metrics validate a tuning change?**  
Elapsed time, correctness, input/shuffle bytes, task-size distribution, spills, CPU/GC, and stage-level skew in Spark UI.

**26. A query has many Exchanges. What next?**  
Find which exchanges are necessary, check reused partitioning, joins/aggregates, data size and distribution; avoid removing shuffles blindly.

### Scenario prompts (Q27–Q30)

**27. Your large-to-large join is slow. What's your approach?**  
Check join type/keys/cardinality and `SortMergeJoin` inputs → Spark UI shuffle/skew/spill → table stats and AQE → consider filtering early, key pruning, pre-aggregation if valid, or distribution changes.

**28. One task takes 10× longer. What's your hypothesis?**  
Skewed data or oversized partition. Check per-task bytes, records, spills and keys; evaluate AQE skew handling and carefully designed salting.

**29. Why is a filter not pushed into Parquet?**  
Unsupported expression (cast/map/UDF), source capability, configuration, or semantics; inspect `PushedFilters`, `DataFilters`, `ReadSchema` and test with a scalar predicate.

**30. Plan looks efficient, but job is slow. Why?**  
Static plans lack actual task metrics; contention, memory/GC, poor file layout, data skew, remote IO, output bottlenecks or under-provisioned executors may dominate.

### Answer structure to practice (60–90 seconds)

**Observation → Operator → Cost → Hypothesis → Change → Measurement.** Example: “The plan shows two `Exchange hashpartitioning` nodes feeding a `SortMergeJoin`. That implies a join shuffle plus sort. I'll inspect actual shuffle volume and skew in the SQL UI, validate join cardinality, and test a broadcast strategy only if one side is genuinely small. Then I'll compare wall time, shuffle metrics and result counts.”

## One-page revision sheet

| Your code | Common physical signature | Interview intuition |
|---|---|---|
| `filter(city == 'boston')` | `Filter`, possible `PushedFilters` | Filter early; check source support |
| `select(...)` / `withColumn(...)` | `Project` | Avoid unnecessary columns; typically narrow |
| `repartition(6)` | `Exchange RoundRobinPartitioning(6)` | Shuffle to redistribute |
| `repartition(4, 'cust_id')` | `Exchange hashpartitioning(cust_id,4)` | Same key together |
| `coalesce(2)` | `Coalesce 2` | Fewer partitions, no *new* shuffle |
| equi-join (both sides large) | `Exchange` + `Sort` + `SortMergeJoin` | Heavy shuffle/sort possible |
| broadcast equi-join | `BroadcastExchange` + `BroadcastHashJoin` | Broadcast small side instead |
| `groupBy('city').count()` | partial/final `HashAggregate` + possible `Exchange` | Local combine before network |
| `groupBy('city').sum('amt')` | partial/final aggregates | Same two-phase concept |
| `groupBy('cust_id').countDistinct('city')` | distinct-key aggregation + exchanges | Global dedup adds work |
| `age.cast('int') > 50` | Spark `Filter`; limited pushdown | Expression/source mismatch |
| AQE enabled | `AdaptiveSparkPlan` | Runtime re-optimization possible |

**Five things to memorize:** `Exchange` means a distribution boundary; `Project` computes/selects fields; partial aggregates save shuffle traffic; pushdown is source dependent; **final performance judgments need Spark UI runtime evidence**.

## Sources, caveats, and what to learn next

### Your supplied sources
- **Transcript:** “How to read Spark query plan”, timestamps **01:35–38:59** (uploaded as `Pasted text(20261102-061058).txt`). Each topic above maps back to its video segment.
- **Starter code:** `2_reading_query_plans.ipynb`, sections on Spark plans, narrow transformations, repartition/coalesce, joins, aggregations, and pushdown. Original relative Parquet paths and image were not part of the upload.

### Official docs used for important corrections and extension
- [Apache Spark: `DataFrame.explain`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.explain.html) — `extended`, `formatted`, `cost`.
- [Apache Spark: `DataFrame.coalesce`](https://spark.apache.org/docs/3.5.6/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.coalesce.html) — narrow dependency and reduction of parallelism.
- [Apache Spark: SQL performance tuning / AQE](https://spark.apache.org/docs/latest/sql-performance-tuning.html) — broadcast joins, AQE and runtime statistics.

**What the source doesn't fully cover:** table statistics accuracy, data skipping/file sizes, nuanced join semantics, bucketing/distribution, `explain` in Spark Connect, and production benchmark methodology. Those are broader interview topics added here as explicitly marked extension material.

### Recommended learning order
1. Re-run §§2–10 without looking at the explanations, and *predict the next operator*.
2. Complete all six exercises and compare to solutions.
3. Explain Q1–Q30 aloud from memory.
4. In Databricks or a cluster, repeat the lab with realistic data and inspect **Spark UI → SQL → executed query → stage metrics**.

**Optional cleanup** (only after you're finished):

In [ ]:
# Deliberately NOT auto-running cleanup: keep data available for re-running earlier cells.
# To delete the temporary data at the end of your study session, uncomment:
# import shutil
# shutil.rmtree(lab_dir, ignore_errors=True)
# spark.stop()
print('Lab artifacts remain available at:', lab_dir)